In [1]:
import subprocess, sys
pkgs = ["opencv-python", "numpy", "pyttsx3"]
subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "-q"])
print("✅ Packages ready")


✅ Packages ready


In [2]:
import cv2
import numpy as np
import pyttsx3
import time
import os

# ── Voice Engine ──────────────────────────────────────
engine = pyttsx3.init()
engine.setProperty('rate', 150)

def speak(text):
    engine.say(text)
    engine.runAndWait()

print("✅ Voice engine ready")


✅ Voice engine ready


In [3]:
# ── Fire Detection (HSV) ─────────────────────────────

def detect_fire_mask(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lower1, upper1 = np.array([0,  120, 200]), np.array([35, 255, 255])
    lower2, upper2 = np.array([35,  50, 200]), np.array([60, 255, 255])
    mask = cv2.bitwise_or(
        cv2.inRange(hsv, lower1, upper1),
        cv2.inRange(hsv, lower2, upper2)
    )
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,   kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel)
    return mask


def calculate_intensity(frame, prev_frame=None):
    mask        = detect_fire_mask(frame)
    fire_pixels = np.sum(mask > 0)
    area_ratio  = fire_pixels / (frame.shape[0] * frame.shape[1])

    gray       = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    brightness = (np.mean(gray[mask > 0]) / 255) if fire_pixels > 0 else 0

    h, _, v    = cv2.split(cv2.cvtColor(frame, cv2.COLOR_BGR2HSV))
    hot_pixels = np.sum((h < 35) & (v > 200) & (mask > 0))
    hot_ratio  = hot_pixels / fire_pixels if fire_pixels > 0 else 0

    flicker = 0.0
    if prev_frame is not None:
        diff_gray = cv2.cvtColor(cv2.absdiff(frame, prev_frame), cv2.COLOR_BGR2GRAY)
        flicker   = np.mean(diff_gray[mask > 0]) / 255 if fire_pixels > 0 else 0

    intensity = 0.5*area_ratio + 0.2*brightness + 0.2*hot_ratio + 0.1*flicker
    return intensity, mask

print("✅ Detection functions ready")


✅ Detection functions ready


## Prediction Logger
 **c** key on the keyboard is pressed to capture a fire frame, the frame + mask + intensity score +  manual label are saved.

### How to label while capturing:
- Once **c** is pressed,  you'll be asked in the terminal: **Is this fire? (y/n)**
- Type y if the frame contains real fire, n if it doesn't
- This becomes the **ground-truth label** for the metrics notebook


In [4]:
# ── Prediction Logger ────────────────────────────────

LOG_FILE = "fire_session_data.npz"

session_frames      = []   # raw BGR frames (resized to 120x160 to save space)
session_masks       = []   # predicted fire masks
session_scores      = []   # predicted intensity scores
session_gt_labels   = []   # YOUR manual labels (1=fire, 0=no-fire)
session_timestamps  = []

def save_session():
    if len(session_frames) == 0:
        print("⚠️  Nothing to save yet.")
        return
    np.savez_compressed(
        LOG_FILE,
        frames      = np.array(session_frames,     dtype=np.uint8),
        masks       = np.array(session_masks,      dtype=np.uint8),
        scores      = np.array(session_scores,     dtype=np.float32),
        gt_labels   = np.array(session_gt_labels,  dtype=np.int8),
        timestamps  = np.array(session_timestamps, dtype=np.float64),
    )
    print(f"💾  Session saved → {LOG_FILE}  ({len(session_frames)} frames)")

print("✅ Logger ready  |  data will be saved to:", os.path.abspath(LOG_FILE))


✅ Logger ready  |  data will be saved to: c:\Users\vansh\Desktop\Fire_Detection\fire_session_data.npz


## Live Webcam Loop

In [5]:
cap   = cv2.VideoCapture(0)
frame1 = None
frame2 = None
stage  = 0
prev_frame = None

print("Controls:")
print("  c  — capture frame (you will be asked to label it)")
print("  r  — reset")
print("  s  — save session data now")
print("  q  — quit & auto-save")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    display = frame.copy()

    if stage == 0:
        cv2.putText(display, "Capture FIRST fire (c)", (10,30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
    elif stage == 1:
        cv2.putText(display, "Capture SECOND fire (c)", (10,30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)

    # Show live intensity on feed
    live_score, live_mask = calculate_intensity(frame, prev_frame)
    cv2.putText(display, f"Live intensity: {live_score:.3f}", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,200,0), 2)

    cv2.imshow("Live Feed", display)
    prev_frame = frame.copy()

    key = cv2.waitKey(1)

    # ── CAPTURE ──────────────────────────────────────
    if key == ord('c'):
        captured = frame.copy()
        score, mask = calculate_intensity(captured)

        # Ask for ground-truth label in terminal
        label_input = input("\n🔥 Is this a FIRE frame? (y=fire / n=no fire): ").strip().lower()
        gt_label = 1 if label_input == 'y' else 0

        # Resize to save space
        small = cv2.resize(captured, (160, 120))
        small_mask = cv2.resize(mask, (160, 120))

        session_frames.append(small)
        session_masks.append(small_mask)
        session_scores.append(float(score))
        session_gt_labels.append(gt_label)
        session_timestamps.append(time.time())

        label_str = "FIRE ✅" if gt_label == 1 else "No Fire ❌"
        print(f"  Logged — intensity={score:.4f}  label={label_str}  total={len(session_frames)} frames")

        # ── Original comparison logic ─────────────────
        if stage == 0:
            frame1 = captured.copy()
            stage  = 1
            speak("First fire captured.")
            cv2.imshow("Fire 1", frame1)

        elif stage == 1:
            frame2 = captured.copy()
            speak("Second fire captured.")
            cv2.imshow("Fire 2", frame2)

            i1, mask1 = calculate_intensity(frame1)
            i2, mask2 = calculate_intensity(frame2, frame1)

            print(f"\nFire 1 Intensity: {i1:.3f}")
            print(f"Fire 2 Intensity: {i2:.3f}")
            cv2.imshow("Mask 1", mask1)
            cv2.imshow("Mask 2", mask2)

            if i1 > i2:
                msg = "Warning! First fire is more intense. Evacuate immediately, take route 2."
            elif i2 > i1:
                msg = "Warning! Second fire is more intense. Evacuate immediately, take route 1."
            else:
                msg = "Both fires have similar intensity."

            print(msg)
            speak(msg)
            stage = 0

    # ── SAVE ─────────────────────────────────────────
    elif key == ord('s'):
        save_session()

    # ── RESET ────────────────────────────────────────
    elif key == ord('r'):
        frame1 = frame2 = None
        stage  = 0
        print("System reset.")
        speak("System reset.")

    # ── QUIT ─────────────────────────────────────────
    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
save_session()   # auto-save on quit
print("\n✅ Session ended. Run fire_detection_metrics.ipynb to evaluate.")


Controls:
  c  — capture frame (you will be asked to label it)
  r  — reset
  s  — save session data now
  q  — quit & auto-save
  Logged — intensity=0.0000  label=No Fire ❌  total=1 frames
  Logged — intensity=0.4242  label=No Fire ❌  total=2 frames

Fire 1 Intensity: 0.000
Fire 2 Intensity: 0.473
Warning! Second fire is more intense. Evacuate immediately, take route 1.
💾  Session saved → fire_session_data.npz  (2 frames)

✅ Session ended. Run fire_detection_metrics.ipynb to evaluate.
